In [8]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Загрузка данных (с проверкой пути к файлу)
file_path = "../data/raw/creditcard.csv"

# Если файл лежит в корне или другой папке, подставьте нужный путь
if not os.path.exists(file_path):
    file_path = "../1_data/creditcard.csv"

df = pd.read_csv(file_path)

# 2. Масштабирование признаков Amount и Time
scaler = StandardScaler()
df["scaled_amount"] = scaler.fit_transform(df[["Amount"]])
df["scaled_time"] = scaler.fit_transform(df[["Time"]])

# Удаляем оригинальные несбалансированные колонки Amount и Time
df_processed = df.drop(["Time", "Amount"], axis=1)

# 3. Разделение на признаки (X) и целевую переменную (y)
X = df_processed.drop("Class", axis=1)
y = df_processed["Class"]

# 4. Stratified Train-Test Split (сохраняем пропорцию 0.17% фрода в обеих выборках)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Обучение Baseline-модели с учетом дисбаланса классов
baseline_model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
baseline_model.fit(X_train, y_train)

# 6. Получение предсказаний
y_pred = baseline_model.predict(X_test)
y_pred_proba = baseline_model.predict_proba(X_test)[:, 1]

# 7. Вывод метрик качества
print("DI DI DI Classification Report DI DI DI")
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraudulent"]))

print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")

DI DI DI Classification Report DI DI DI
              precision    recall  f1-score   support

  Legitimate       1.00      0.98      0.99     56864
  Fraudulent       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962

ROC-AUC Score: 0.9722
